In [1]:
#Import bibliotecas
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, count, col, current_timestamp, from_json, lit, row_number
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.window import Window

In [2]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [4]:
#Iniciando sessão spark
spark = SparkSession.builder \
    .appName("bronze-layer") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6229ebf5-32cd-40b8-bfde-553526bfa799;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
:: resolution report :: resolve 366ms :: artifacts dl 8ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 from central in [default]
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number

In [5]:
!ls /tmp/warehouse/bronze/transacoes

data  metadata


In [6]:
#Variaveis de conexao ao ambiente postgres
conexao = {
           "url": "jdbc:postgresql://postgres:5432/pipeline_transacoes",
           "user": "postgres",
           "password": "1234",
           "driver": "org.postgresql.Driver"
}

In [7]:
#Dados vindo da API
df_raw = spark.read.jdbc(
    url=conexao["url"],
    table="staging.transacoes_raw",  # tabela no PostgreSQL
    properties={
        "user": conexao["user"],
        "password": conexao["password"],
        "driver": conexao["driver"]
    }
)

df_raw.select("payload").show(truncate=True)

+--------------------+
|             payload|
+--------------------+
|{"valor": 2437.22...|
|{"valor": 775.25,...|
|{"valor": 528.18,...|
|{"valor": 2253.03...|
|{"valor": null, "...|
|{"valor": 1942.65...|
|{"valor": 3931.11...|
|{"valor": 2090.21...|
|{"valor": 4579.7,...|
|{"valor": 4995.86...|
|{"valor": 1422.14...|
|{"valor": 4902.43...|
|{"valor": 1797.08...|
|{"valor": 1114.42...|
|{"valor": 2306.02...|
|{"valor": 2266.53...|
|{"valor": 1972.3,...|
|{"valor": 54.76, ...|
|{"valor": null, "...|
|{"valor": 3382.19...|
+--------------------+
only showing top 20 rows



In [8]:
#Criando tabela bronze se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

#Definir schema do payload
schema_payload = StructType([
    StructField("valor", DoubleType(), True),
    StructField("produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("metodo_pagamento", StringType(), True),
])

# Transformar DF raw
df_bronze = (
    df_raw
    .withColumn("data_ingestao", current_timestamp())
    .withColumn("data_particao", to_date(col("data_ingestao")))
    .withColumn("payload_struct", from_json(col("payload"), schema_payload))
    .select(
        "id",
        "payload_struct.*",
        "data_ingestao",
        "data_particao"
    )
)                           
df_bronze.writeTo("lakehouse.bronze.transacoes") \
         .using("iceberg") \
         .partitionedBy("data_particao") \
         .createOrReplace()

In [9]:
#Select tabela transacoes na bronze
spark.sql("SELECT * FROM lakehouse.bronze.transacoes LIMIT 5").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:02:...|   2026-04-12|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:02:...|   2026-04-12|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:02:...|   2026-04-12|
|  4|2253.03| celular|eletronicos|        53|         2|          boleto|2026-04-12 17:02:...|   2026-04-12|
|  5|   NULL|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:02:...|   2026-04-12|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+



In [10]:
#Validando estrutura tabela bronze
spark.sql("Describe lakehouse.bronze.transacoes").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|                  id|      int|   NULL|
|               valor|   double|   NULL|
|             produto|   string|   NULL|
|           categoria|   string|   NULL|
|          cliente_id|      int|   NULL|
|          quantidade|      int|   NULL|
|    metodo_pagamento|   string|   NULL|
|       data_ingestao|timestamp|   NULL|
|       data_particao|     date|   NULL|
|# Partition Infor...|         |       |
|          # col_name|data_type|comment|
|       data_particao|     date|   NULL|
+--------------------+---------+-------+



In [11]:
#Validando a existencia de dados nulos na bronze
from pyspark.sql.functions import col, sum
df_bronze.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_bronze.columns
]).show()

+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
|  0|   10|      0|        0|         0|         0|               0|            0|            0|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+



In [12]:
#Verificando dados nulos no campo VALOR
spark.sql("SELECT * FROM lakehouse.bronze.transacoes where valor is null").show()

+---+-----+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-----+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  5| NULL|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:02:...|   2026-04-12|
| 19| NULL|    fone|eletronicos|        74|         2|          boleto|2026-04-12 17:02:...|   2026-04-12|
| 23| NULL|notebook|eletronicos|         6|         1|             pix|2026-04-12 17:02:...|   2026-04-12|
| 27| NULL|camiseta|  vestuario|        32|         1|          boleto|2026-04-12 17:02:...|   2026-04-12|
| 31| NULL| celular|eletronicos|        16|         3|             pix|2026-04-12 17:02:...|   2026-04-12|
| 36| NULL|notebook|eletronicos|        97|         1|          boleto|2026-04-12 17:02:...|   2026-04-12|
| 40| NULL| celular|eletronicos|     

In [13]:
#VALIDAR DUPLICATAS NA BRONZE
df_bronze = spark.read.table("lakehouse.bronze.transacoes")

df_bronze.groupBy("id", "data_ingestao") \
    .agg(count("*").alias("qtd")) \
    .filter(col("qtd") > 1) \
    .show()

+---+-------------+---+
| id|data_ingestao|qtd|
+---+-------------+---+
+---+-------------+---+



In [14]:
#  TRANSFORMAÇÕES (SILVER)
# - tratamento de null
# - deduplicação
# - nova coluna derivada
window = Window.partitionBy("id", "data_ingestao").orderBy(col("data_ingestao").desc())

df_curated = (
    df_bronze
    .fillna({"valor": 0})  # tratamento de null
    .withColumn("row_num", row_number().over(window))  # deduplicação
    .filter(col("row_num") == 1)
    .drop("row_num")
    .withColumn("preco_unitario", col("valor") / col("quantidade"))  # transformação extra
)

# Criar database
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.silver")

# Salvar
df_curated.writeTo("lakehouse.silver.transacoes") \
          .using("iceberg") \
          .partitionedBy("data_particao") \
          .createOrReplace()

spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:02:...|   2026-04-12|           1218.61|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:02:...|   2026-04-12| 258.4166666666667|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:02:...|   2026-04-12|176.05999999999997|
|  4|2253.03| celular|eletronicos|        53|         2|          boleto|2026-04-12 17:02:...|   2026-04-12|          1126.515|
|  5|    0.0|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:02:...|   2026-04

In [15]:
df_curated.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()

In [16]:
print(" Validando duplicatas após tratamento...")

df_curated.groupBy("id", "data_ingestao") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print(" Dados finais na Silver:")
spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

 Validando duplicatas após tratamento...
+---+-------------+-----+
| id|data_ingestao|count|
+---+-------------+-----+
+---+-------------+-----+

 Dados finais na Silver:
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:02:...|   2026-04-12|           1218.61|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:02:...|   2026-04-12| 258.4166666666667|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:02:...|   2026-04-12|176.05999999999997|
|  4|2253.03| celular|eletronicos|        53|         2|     

In [17]:
#Atualização na tabela silver (ADD COLUMNS)
df_add = df_curated.withColumn("preco_unitario", (col("valor") / col("quantidade")))
df_add.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()

In [18]:
#Validando após alteração
spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|2437.22|notebook|eletronicos|        67|         2|          boleto|2026-04-12 17:02:...|   2026-04-12|           1218.61|
|  2| 775.25|    fone|eletronicos|        50|         3|             pix|2026-04-12 17:02:...|   2026-04-12| 258.4166666666667|
|  3| 528.18| celular|eletronicos|         1|         3|             pix|2026-04-12 17:02:...|   2026-04-12|176.05999999999997|
|  4|2253.03| celular|eletronicos|        53|         2|          boleto|2026-04-12 17:02:...|   2026-04-12|          1126.515|
|  5|    0.0|notebook|eletronicos|        76|         3|  cartao_credito|2026-04-12 17:02:...|   2026-04

In [19]:
!ls /tmp/warehouse/bronze/transacoes/data/data_particao=2026-04-12

00000-2-4079e7d3-717a-462e-8a05-7615e9019dbb-00001.parquet
00000-2-57ff789e-e4de-4480-8b25-bdd07844e6a5-00001.parquet
00000-2-6c7cc06c-9cfc-4b99-85de-436552f31d11-00001.parquet
00000-2-a924c076-98c1-4f89-936a-ffde7769d073-00001.parquet
00000-2-e1b5390c-fce2-4f2e-a324-74857c87552a-00001.parquet
00000-8-8396e66f-e1db-4ba4-8e7f-7f62199a6118-00001.parquet
00000-9-19d438e3-7eb4-498a-abe7-069fa9113305-00001.parquet
